In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# Inputs

In [ ]:
local = "sm"
year = "2026"
month = "01"

In [ ]:
df = pd.read_excel(f"data/{local}-sales-{year}-{month}.xls", sheet_name="Adiciones")
df_expenses = pd.read_excel(f"data/{local}-gastos-{year}-{month}.xlsx", sheet_name="Gastos", skiprows=3)

In [ ]:
df_expenses.head(5).to_clipboard(index=False)

# Read Sales

In [ ]:
def sales_clean_up_data(df) -> pd.DataFrame:
    skipped_columns = [
        "Id. Venta",
        "Creación",
        "Producto",
        "Categoría",
        "Cantidad",
        "Precio",
        "Costo base",
        "Costo modificadores",
        "Costo total",
        "Creada por",
    ]
    df = df[skipped_columns]

    # if Producto column is 'Duo Familiar (2pizzas)' set Costo modificadores to 0
    df.loc[df["Producto"] == "Duo Familiar (2pizzas)", "Costo modificadores"] = 0

    df["precio_unitario"] = df["Precio"] / df["Cantidad"]
    df["costo_unitario"] = df["Costo base"] / df["Cantidad"]
    df["ingreso"] = df["precio_unitario"] * df["Cantidad"]

    df["comision"] = 0.0
    df.loc[df["Creada por"] == "uber_eats", "comision"] = df["ingreso"] * 0.30

    df["Costo total"] = (df["costo_unitario"] * df["Cantidad"]) + df["Costo modificadores"] + df["comision"]

    df["ingreso"] = df["ingreso"].fillna(0).astype(float)
    df["ingreso_sin_iva"] = df["ingreso"] / 1.19

    df["margen"] = df["ingreso"] - df["Costo total"]

    df["margen_sin_iva"] = df["ingreso_sin_iva"] - df["Costo total"]

    df["created_at"] = pd.to_datetime(df["Creación"], errors='coerce')


    df.drop(columns=["Creación"], inplace=True)
    df.sort_values(by="created_at", inplace=True)

    df.to_excel(f"data/{local}-sales-{year}-{month}-processed.xlsx", index=False)
    return df

In [ ]:
df = sales_clean_up_data(df=df)

# Expenses

In [ ]:
df_expenses

# Salary

In [ ]:
# sueldos_enero = 4_781_837 # CG
sueldos_enero = 0 # SM included in expenses

# Results

In [ ]:
def kpi_calculations(df: pd.DataFrame, df_expenses: pd.DataFrame) -> str:  
    total_margen = df["margen"].sum()
    total_ingreso = df["ingreso"].sum()
    comision_total = df["comision"].sum()
    costo_total = df["Costo total"].sum()
    total_margen_sin_iva = df["margen_sin_iva"].sum()

    gastos_totales = df_expenses["Importe"].sum()
    # sum all values in Importe column where "Estado del pago" is "Pagado"
    pagados_totales = df_expenses[df_expenses["Estado del pago"] == "Pagado"]["Importe"].sum()
    por_pagar_totales = df_expenses[df_expenses["Estado del pago"] == "A pagar"]["Importe"].sum()

    ebitda = total_margen_sin_iva - gastos_totales - sueldos_enero
    ebitda_percentage = (ebitda / total_ingreso) * 100

    output = f"""
        Ingresos
        total_margen: {total_margen:,.0f}
        total_ingreso: {total_ingreso:,.0f}
        comision_total: {comision_total:,.0f}
        costo_total: {costo_total:,.0f}
        total_margen_sin_iva: {total_margen_sin_iva:,.0f}

        Gastos
        gastos_totales: {gastos_totales:,.0f}
        pagados_totales: {pagados_totales:,.0f}
        por_pagar_totales: {por_pagar_totales:,.0f}"

        EBITDA
        ebitda: {ebitda:,.0f}
        ebitda_percentage: {ebitda_percentage:.2f}%
        """
    return output

In [ ]:
output = kpi_calculations(df=df, df_expenses=df_expenses)
print(output)